# MerLin Release 0.4 Highlights

We are presenting the new features and changes introduced in MerLin 0.4.x. 

## 0. Imports

In [1]:
import merlin as ml
import perceval as pcvl
import torch
from merlin.datasets import iris
from merlin.core import StateVector

## 1. New Features

Here is a brief overview of the main new features.

### 1.1 ``ReservoirClassifier``

ReservoirClassifier is the new ready-to-use QORC model. It builds a frozen photonic reservoir from a Haar-random interferometer and trains only a classical linear readout.

This makes the QORC workflow available as a reusable model instead of requiring users to manually wire preprocessing, reservoir feature extraction, caching, and readout training. The model supports optional scikit-learn dimensionality reduction, reservoir feature normalization, cached embeddings, and direct creation of PyTorch datasets for the readout.

For more details, checkout the [ReservoirClassifier documentation](../../user_guide/models/reservoir_classifier.rst).

The basic workflow of this new object will be presented as we try to classify the Iris dataset.

In [2]:
# 0. Get the data
train_features, train_labels, train_metadata = iris.get_data_train()
test_features, test_labels, test_metadata = iris.get_data_test()

# 1. Create the reservoir classifier
#TODO Remove when the import is fixed
from merlin.models.reservoir_classifier import ReservoirClassifier
reservoir_classifier=ReservoirClassifier(in_features=4,out_features=3,n_photons=2)

# 2. Call fit reservoir on the training inputs
reservoir_classifier.fit_reservoir(train_features)

# 3. Generate the reservoir embeddings
train_embeddings = reservoir_classifier.transform_reservoir(train_features)
test_embeddings = reservoir_classifier.transform_reservoir(test_features)
## make_dataset(X, y) can also be used to return a TensorDataset instead
# of a simple tensor

## If you want the output logits of the reservoir,call predict(X)

# 4. Train the readout layer
readout = torch.nn.Linear(
    train_embeddings.shape[1],
    3,
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(readout.parameters(), lr=0.01)

for epoch in range(20):
    optimizer.zero_grad()

    logits = readout(train_embeddings)
    loss = criterion(logits, torch.tensor(train_labels))

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}: loss={loss.item():.4f}")

# 5. Evaluate the model
with torch.no_grad():
    logits = readout(test_embeddings)
    predictions = logits.argmax(dim=1)

accuracy = (predictions == test_labels).float().mean().item()

print(f"Test accuracy: {accuracy:.4f}")

Epoch 1: loss=1.3705
Epoch 2: loss=1.3024
Epoch 3: loss=1.2373
Epoch 4: loss=1.1752
Epoch 5: loss=1.1163
Epoch 6: loss=1.0607
Epoch 7: loss=1.0083
Epoch 8: loss=0.9591
Epoch 9: loss=0.9131
Epoch 10: loss=0.8702
Epoch 11: loss=0.8302
Epoch 12: loss=0.7929
Epoch 13: loss=0.7583
Epoch 14: loss=0.7260
Epoch 15: loss=0.6961
Epoch 16: loss=0.6683
Epoch 17: loss=0.6424
Epoch 18: loss=0.6182
Epoch 19: loss=0.5957
Epoch 20: loss=0.5747
Test accuracy: 0.7667


### 1.2 ``PhotonicGenerator``

`PhotonicGenerator` is a PyTorch module for latent-to-sample generative
workflows. It wraps one or more `QuantumLayer` heads and delegates the final
classical interpretation to an output adapter.

This provides a cleaner base for photonic GAN-style generators. Instead of manually feeding latent variables into a quantum layer and reshaping raw measurement distributions for each experiment, users can define a generator with a latent dimension, one or more quantum heads, and an adapter such as `ImageAdapter` or `VectorAdapter`.

The generator exposes `sample_latent`, `measure`, `generate`, and normal PyTorch `forward` behavior.

For more details, checkout the [QGAN documentation](../../user_guide/models/qgan.rst).

Below is some examples on how to instantiate a `PhotonicGenerator`.

In [3]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.FOCK,
    ),
)


#Option 1: Same quantum layer repeated for each generator head
generator = ml.PhotonicGenerator(
    layers=qlayer,
    count=2,    #Number of henerator heads
    # Adapter of the output of the layer to the desired data format. 
    # Here it takes the layers' ouput and transforms it to an image
    output_adapter=ml.ImageAdapter(     
        shape=(1, 32, 32),  #Shape of one image
        headwise=True,
        normalize_patches=True,
    ),
)
print(generator)

#Option 2: Same quantum layer repeated for each generator head
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

qlayer_2 = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.FOCK,
    ),
)


generator = ml.PhotonicGenerator(
    layers=[qlayer,qlayer_2], #Sequence of layers each being a generator
    # Adapter of the output of the layer to the desired data format. 
    # Here it takes the layers' ouput and transforms it to a vector f the correct size
    output_adapter=ml.VectorAdapter(
        size=32 #Desired size
    ),
)
print(generator)

PhotonicGenerator(
  (layers): ModuleList(
    (0-1): 2 x QuantumLayer(
      (_photon_loss_transform): PhotonLossTransform()
      (_detector_transform): DetectorTransform()
      (measurement_mapping): Probabilities()
    )
  )
  (output_adapter): ImageAdapter(
    (_vector_adapter): VectorAdapter()
  )
)
PhotonicGenerator(
  (layers): ModuleList(
    (0-1): 2 x QuantumLayer(
      (_photon_loss_transform): PhotonLossTransform()
      (_detector_transform): DetectorTransform()
      (measurement_mapping): Probabilities()
    )
  )
  (output_adapter): VectorAdapter()
)


### 1.3. ``QCNNClassifier``

`QCNNClassifier` adds a staged photonic QCNN model for image-like inputs. Architectures are built from validated `QConv`, `QPool`, and `QDense` stages. Users can provide their own stage sequence or rely on the default validated architecture.

This gives users a higher-level image-classification model without requiring them to assemble every quantum convolution, pooling step, and dense readout by
hand.

For more details, checkout the [QCNN documentation](../../user_guide/models/qcnn.rst).

Below is some examples on how to instantiate a `QCNNClassifier`.

In [4]:
qcnn = ml.QCNNClassifier(
    input_shape=(16,16),   #Shape of the input
    num_classes=3,  #Number of classes
    #Stages in QCNN classifier, this is the default values when stages=None
    stages= [
        ml.QCNNClassifier.QConv(kernel_size=2, stride=2),
        ml.QCNNClassifier.QPool(kernel_size=2),
        ml.QCNNClassifier.QDense()
        ],
    )

print(qcnn)

QCNNClassifier(
  (layers): Sequential(
    (QConv_1): QuantumLayer(
      (_photon_loss_transform): PhotonLossTransform()
      (_detector_transform): DetectorTransform()
      (measurement_mapping): Amplitudes()
    )
    (QPool_1): QuantumLayer(
      (_photon_loss_transform): PhotonLossTransform()
      (_detector_transform): DetectorTransform()
      (measurement_mapping): Identity()
    )
    (QDense): QuantumLayer(
      (_photon_loss_transform): PhotonLossTransform()
      (_detector_transform): DetectorTransform()
      (measurement_mapping): Probabilities()
    )
    (Readout): Linear(in_features=136, out_features=3, bias=True)
  )
)


### 1.4. Memristive Phase Shifters

`CircuitBuilder` now supports memristive phase shifters through
`add_memristive_ps(...)`.

For machine-learning workflows, this gives a photonic layer a stateful
component that can carry information across repeated calls or timesteps. That is useful for temporal dependence, recurrent-style models, feedback circuits, and sequence experiments where the current circuit response should depend on previous inputs rather than only on the current tensor.

The builder declares the memristive component, and `QuantumLayer` owns the runtime state, history, serialization hooks, reset behavior, and
`detach_memristive_state(...)` helper. This moves a previously manual feedback layer pattern into the normal builder-layer contract.

For more information on the use of the memristive phase shifters, please consult the corresponding section in [QuantumLayer's documentation](../../api_reference/api/merlin.algorithms.layer.rst).

Here is a quick example on how to use a memristive phase shifter.

In [5]:
# 1. Define the update rule that the phase shifter will follow. 
# It must have this sugnature but output may be a typed object
# since it is directly the return of the QuantumLayer.
def update_rule_exp(state: torch.Tensor, output: torch.Tensor):
    return torch.exp(state + output[:, 0])

#2. Create the layer with the circuit builder
circ = ml.CircuitBuilder(n_modes=3)
circ.add_entangling_layer()
# Add a memristive phase shifter
circ.add_memristive_ps(
    mode=0, #The mode onto which the memristive phase shifter must be applied
    update_rule=update_rule_exp, #The update rule of the phase shifter defined in 1
    initial_state=0.01  #The initial value of the phase shifter
)
circ.add_entangling_layer()
circ.add_angle_encoding(modes=[0, 2])
circ.add_entangling_layer()

#3. Create the quanrum layer
ql = ml.QuantumLayer(
    builder=circ,
    n_photons=3,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.FOCK
    ),
)
#3.5 Rest the memristor to the correct batch size
ql.reset(batch_size=2)
print(f"Memristive state before the forward pass {ql.memristive_state}")
print(f"Memristive history before the forward pass {ql.memristive_history}")
print()

#4. Do a forward pass
output=ql(torch.rand((2,2)))
print(f"Memristive state after the forward pass {ql.memristive_state}")
print(f"Memristive history after the forward pass {ql.memristive_history}")

Memristive state before the forward pass [tensor([0.0100, 0.0100])]
Memristive history before the forward pass [[tensor([0.0100, 0.0100])]]

Memristive state after the forward pass [tensor([1.1227, 1.1382])]
Memristive history after the forward pass [[tensor([0.0100, 0.0100]), tensor([1.1227, 1.1382])]]


### 1.5 Noisy SLOS Simulations

`QuantumLayer` now supports a noisy SLOS path driven by `pcvl.NoiseModel`. MerLin can account for the main hardware-relevant noise sources directly in simulation

- brightness;
- transmittance;
- phase imprecision;
- phase error;
- indistinguishability;
- multi-photon emission through `g2`;
- distinguishability of extra `g2` photons.

For more information on the significance of each noise source, please consult the [following documentation](../../user_guide/noisy_simulations.rst).

Noisy simulations return probabilities. Source noise and stochastic phase errors are represented as probability mixtures, so MerLin combines probability distributions rather than returning ideal amplitudes.

For `g2` and source-noise cases, outputs may span multiple photon-number sectors. These outputs are represented through `SectoredDistribution` and `SectorResult`, then converted back into tensors when required by the selected measurement strategy.

For details on the actual implementation of these noisy simulation please consult [this documentation](../../quantum_expert_area/noisy_simulations.rst)

#### Why noisy simulations are useful?

Current photonic processors are noisy. A model trained only with ideal
simulation can therefore learn from output probabilities that are cleaner than the probabilities produced by the hardware.

Merlin can add the main Quandela hardware noise sources to SLOS simulations through :class:`pcvl.NoiseModel`. This makes the training distribution closer to the distribution expected from the QPU.

#### Simulate the actual results of the Ascella computer

Here we will compare the perfect simulation of a simple quantum layer to a noisy one and compare the results. Here are the Ascella publicly available noise statistics at the time of the v0.4.0 release

- ``indistinguishability`` = 0.8636
- ``transmittance`` = 0.0244
- ``g2`` = 0.01955

In [6]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

#Non noisy layer
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.UNBUNCHED,
    ),
)


# Noisy layer
## Option 1 pass the noise to a perceval experiment and then to the layer
experiment=pcvl.Experiment(
    m_circuit=circuit.to_pcvl_circuit(),
   noise=pcvl.NoiseModel(
       indistinguishability=0.8636,
       transmittance=0.0244,g2=0.01955
       )
    )
noisy_qlayer = ml.QuantumLayer(
    input_size=2,
    experiment=experiment,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.UNBUNCHED,
    ),
    input_parameters=["px"],
    trainable_parameters=["el"],
)
## Option 2 pass the noise to the noise argument
noisy_qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.UNBUNCHED,
    ),
    noise=pcvl.NoiseModel(
        indistinguishability=0.8636,
        transmittance=0.0244,
        g2=0.01955
    )
)

#Run the layers
non_noisy_output=qlayer(torch.tensor([0.5,1.1]))
noisy_output=noisy_qlayer(torch.tensor([0.5,1.1]))

print(f"The perfect simulation has output:")
for key, prob in zip(qlayer.output_keys, non_noisy_output.flatten()):
        print(f"Output probability of state {key} is {prob}")
print()

print(f"The noisy simulation has output:")
for key, prob in zip(noisy_qlayer.output_keys, noisy_output.flatten()):
        print(f"Output probability of state {key} is {prob}")

The perfect simulation has output:
Output probability of state (1, 0, 0) is 0.25901803374290466
Output probability of state (0, 1, 0) is 0.2976556122303009
Output probability of state (0, 0, 1) is 0.44332635402679443

The noisy simulation has output:
Output probability of state (1, 0, 0) is 0.01715298369526863
Output probability of state (0, 0, 0) is 0.9753625988960266
Output probability of state (0, 1, 0) is 0.006604863330721855
Output probability of state (0, 0, 1) is 0.0008735398296266794
Output probability of state (2, 0, 0) is 2.8784731966879917e-06
Output probability of state (1, 1, 0) is 2.216747816419229e-06
Output probability of state (1, 0, 1) is 2.931805340722349e-07
Output probability of state (0, 2, 0) is 4.2678627210079867e-07
Output probability of state (0, 1, 1) is 1.1289098722500057e-07
Output probability of state (0, 0, 2) is 7.465315476906653e-09


/Users/lfvigneux/Documents/GitHub/merlin/merlin/utils/deprecations.py:545: UserWarning: Noisy simulations with source noise currently use ComputationSpace.FOCK. Other computation spaces are not yet supported for noise models. pcvl.detectors can be used to use custom post-selection.
  return func(*f_args, **kwargs)


We observe that the outputs are very different. So training locally your quantum models with noise models that are similar to the actual QPU will be useful to get meaningful QPU results.

### 1.6 New ``EncodingSpace`` Object

`EncodingSpace` describes how the last dimension of an amplitude tensor maps into MerLin's canonical Fock basis. It makes logical amplitude inputs explicit and removes the need to manually enumerate Fock states for common encodings.

Supported encodings include:

- `EncodingSpace.FOCK` for amplitudes already in full Fock order;
- `EncodingSpace.UNBUNCHED` for collision-free photonic states;
- `EncodingSpace.DUAL_RAIL` for qubit-like binary logical states;
- partitioned encodings through `EncodingSpace(modes_per_photon=[...]`;
- QLOQ-style grouped encodings through `EncodingSpace.qloq(...)`.

`StateVector.from_tensor(..., encoding=...)` validates the logical tensor, embeds it into Fock order, and returns a `StateVector` that can be passed to `QuantumLayer.forward()`.

For more details, consult the [API documentation](../../api_reference/api/merlin.core.encoding_space.rst)

Here is an example of the usage for a tensor in the unbunched space of 2 photons and 4 modes.

In [7]:
#1. Define the tensor in the logical space
#There is 6 basis states in the unbunched space of 2 photons and 4 modes
unbunched_tensor=torch.arange(1,7,dtype=torch.float64)
unbunched_tensor=unbunched_tensor/torch.norm(unbunched_tensor)
print(unbunched_tensor)

#2. Encode it into a state vector with the EncodingSpace object
encoded_state=StateVector.from_tensor(tensor=unbunched_tensor,encoding=ml.EncodingSpace.UNBUNCHED,n_modes=4,n_photons=2)
print(encoded_state.tensor)

tensor([0.1048, 0.2097, 0.3145, 0.4193, 0.5241, 0.6290], dtype=torch.float64)
tensor([0.0000+0.j, 0.1048+0.j, 0.2097+0.j, 0.3145+0.j, 0.0000+0.j, 0.4193+0.j, 0.5241+0.j,
        0.0000+0.j, 0.6290+0.j, 0.0000+0.j])


### 1.7 MerlinProcessor Local Processor Support


`MerlinProcessor` now accepts Perceval `AProcessor` objects through the keyword-only `processor=` argument.

This means local Perceval processors can be used from the same Merlin execution interface as remote processors and sessions. Local processors use a local backend route. `RemoteProcessor` instances passed through `processor=` are normalized to the remote backend route. The older `remote_processor=` argument still works as a deprecated compatibility path.

This is useful when users want to evaluate Merlin models on local Perceval backends, including local noisy or sampling backends, without leaving the Merlin model execution workflow.

Checkout the corresponding [ deprecation section](#31-remote_processor-argument-of-the-merlinprocessor-is-deprecated) explaining the migration.

### 1.8 Reclarified kernel contract

The kernel has been restructured to be built directly on ``QuantumLayer``s instead of a completely different backbone. That way, the updates and changes of the ``QuantumLayer`` are directly applied to the kernels. Here are the new contracts.

- `FeatureMap` – a descriptor that stores
	 the photonic circuit and its parameter layout. It accepts:

	 - a `pcvl.Circuit` (manual construction),
	 - a `CircuitBuilder` (declarative), or
	 - a `pcvl.Experiment` (unitary circuit + measurement semantics).


- `FidelityKernel` – validates the feature map, builds the intrinsic ``QuantumLayer``, normalizes public inputs, and delegates kernel-matrix construction to the backend.


The compability methods such as `KernelCircuitBuilder`,
`FidelityKernel.simple(...)` and simple-factory compatibility
arguments are deprecated since the ``QuantumLayer`` backbone is now used.

### 1.9 StateVector and Sparse State Workflows

`StateVector` and probability-distribution objects have improved support for sparse tensors and logical-to-Fock mappings. This helps amplitude-input workflows stay memory-aware, especially when the logical state is much smaller than the full Fock basis.

## 2. Breaking Changes
Here are features that are now completely removed from v.0.4.

### 2.1 The ``no_bunching`` Flag Is Now Removed

The ``no_bunching`` flag used in version 0.2 and 0.2 in many functions (QuantumLayer and kernels definitions) was deprecated since version 0.3.0 and is now removed. The new way of deciding to use the unbunched or full Fock computation space is with the ``ComputationSpace`` object in the ``MeasurementStrategy``. Here we present the two ways to define a ``QuantumLayer`` with the equivalent of settng the ``no_bunching`` flag to True or False.

In [8]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old flag, breaking change

In [9]:
# # Unbunched space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     no_bunching=True
#     )

# # Full Fock space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     no_bunching=False
#     )

Equivalents of the flag

In [10]:
# Equivalent of no_bunching=True
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.UNBUNCHED,
    ),
)
print(qlayer.computation_space)

## Or, because the unbunched space is applied by default
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
)
print(qlayer.computation_space)

# Equivalent of no_bunching=False
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.FOCK,
    ),
)
print(qlayer.computation_space)

ComputationSpace.UNBUNCHED
ComputationSpace.UNBUNCHED
ComputationSpace.FOCK


### 2.2 Constructor ``computation_space`` Is Removed

The legacy no_bunching flag is no longer accepted on public layer and kernel entry points. It was deprecated in v.0.3 and is now removed in v.0.4.

Use explicit computation-space configuration instead:

In [11]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old flag, breaking change

In [12]:
# # Unbunched space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     computation_space=ml.ComputationSpace.UNBUNCHED
#     )

# # Full Fock space
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     computation_space=ml.ComputationSpace.FOCK,
#     )

Equivalents of the flag

In [13]:
# Equivalent of computation_space=ml.ComputationSpace.UNBUNCHED
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.UNBUNCHED,
    ),
)
print(qlayer.computation_space)

# Equivalent of computation_space=ml.ComputationSpace.FOCK
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(
        computation_space=ml.ComputationSpace.FOCK,
    ),
)
print(qlayer.computation_space)

ComputationSpace.UNBUNCHED
ComputationSpace.FOCK


### 2.3 Legacy MeasurementStrategy Access Is Removed

Legacy enum-style measurement access such as MeasurementStrategy.PROBABILITIES, MeasurementStrategy.MODE_EXPECTATIONS, and MeasurementStrategy.AMPLITUDES now fails with a migration error. Passing measurement strategies as strings, such as "PROBABILITIES", is also no longer supported.

The accepted way is now to use ``MeasurementStrategy`` factory methods:

- ``MeasurementStrategy.probs(...)``
- ``MeasurementStrategy.mode_expectations(...)``
- ``MeasurementStrategy.amplitudes(...)``
- ``MeasurementStrategy.partial(...)``

Here are the suggested replacements

- ``MeasurementStrategy.PROBABILITIES``
    - ``MeasurementStrategy.probs(computation_space=...)``
- ``MeasurementStrategy.MODE_EXPECTATIONS``
    - ``MeasurementStrategy.mode_expectations(computation_space=...)``
- ``MeasurementStrategy.AMPLITUDES``
    - ``MeasurementStrategy.amplitudes(computation_space=...)``
- ``MeasurementStrategy.NONE``
    - ``MeasurementStrategy.amplitudes(computation_space=...)``
- ``"PROBABILITIES"`` (string)
    - ``MeasurementStrategy.probs(computation_space=...)``

We will present the migration for the probabilities strategy.

In [14]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old accesses, breaking change

In [15]:
# # Enum-style access
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     measurement_strategy=ml.MeasurementStrategy.PROBABILITIES
#     )

# # String access
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     measurement_strategy="PROBABILITIES"
#     )

Equivalents of the flag

In [16]:
# Equivalent of no_bunching=True
qlayer = ml.QuantumLayer(
    input_size=2,
    builder=circuit,
    n_photons=1,
    measurement_strategy=ml.MeasurementStrategy.probs(),
)
print(qlayer.measurement_strategy)

MeasurementStrategy(type=<MeasurementKind.PROBABILITIES: 'PROBABILITIES'>, measured_modes=(), computation_space=<ComputationSpace.UNBUNCHED: 'unbunched'>, grouping=None, occupancy_readout=False)


### 2.4. Constructor Amplitude Encoding Removed

``QuantumLayer(..., amplitude_encoding=True)`` is no longer accepted.

Amplitude inputs should now be passed to ``QuantumLayer.forward()`` as either a ``StateVector`` or a complex tensor.

Let's present these alternatives.

In [17]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()

Old flag, breaking change

In [18]:
# # Enum-style access
# qlayer=ml.QuantumLayer(
#     input_size=2,
#     builder=circuit,
#     n_photons=1,
#     amplitude_encoding=True
#     )

Equivalents of the flag

In [19]:
qlayer=ml.QuantumLayer(
    builder=circuit,
    n_photons=1,
    )

# Option 1: StateVector input
input_state = StateVector.from_tensor(
    tensor=torch.rand(1, 3),
    n_modes=3,
    n_photons=1,
    encoding=ml.EncodingSpace.FOCK,
)
qlayer(input_state)
# Option 2: complex tensor
input_state = torch.rand(1, qlayer.output_size, dtype=torch.complex64)

qlayer(input_state)

tensor([[0.1356, 0.6801, 0.1843]], grad_fn=<ViewBackward0>)

### 2.5 Tensor Constructor ``input_state`` Removed

Passing a raw ``torch.Tensor`` as constructor input_state is no longer accepted. If a state object is needed at construction time, build it explicitly with ``StateVector.from_tensor(...)``.

Here is exactly how to migrate from the previous API.

In [20]:
# Define the basic interferometer
circuit = ml.CircuitBuilder(n_modes=3)
circuit.add_entangling_layer()
circuit.add_angle_encoding([0, 1])
circuit.add_entangling_layer()

Old flag, breaking change

In [21]:
# # Tensor input state
# amplitudes=torch.rand(2)
# qlayer=ml.QuantumLayer(
#     input_size=3,
#     builder=circuit,
#     n_photons=1,
#     input_state=amplitudes
#     )

Equivalents of the flag

In [22]:
# StateVector input
amplitudes=torch.rand(3)
input_state=StateVector.from_tensor(
      amplitudes,
      n_modes=3,
      n_photons=1,
      encoding=ml.EncodingSpace.UNBUNCHED,
  )

qlayer=ml.QuantumLayer(
    input_size=2,
    builder=circuit,n_photons=1,
    input_state=input_state
    )

## 3. Deprecations

Two new deprecations are effective since version v.0.4.

### 3.1 ``remote_processor`` Argument of the ``MerlinProcessor`` Is Deprecated

The name of the previous ``remote_processor`` argument is now simply ``processor`` where you pass the same Perceval processor.

In [23]:
#Deprecated
# local_processor = pcvl.Processor("SLOS")
# proc = ml.MerlinProcessor(remote_processor=local_processor)

#New method
local_processor = pcvl.Processor("SLOS")
proc = ml.MerlinProcessor(processor=local_processor)

### 3.2 Kernel Compatibility Helpers Are Deprecated

As it was identifed in the [new features section](#18-reclarified-kernel-contract), kernel compatibility helpers such as ``KernelCircuitBuilder``, ``FidelityKernel.simple(...)``, and related simple-factory compatibility arguments are deprecated and kept only for transition.

## 4. Compatibility and Maintenance

MerLin 0.4 updates the dependency floor to ``perceval-quandela>=1.2.1`` and allows newer ``PyTorch`` versions up to ``torch<=2.12.0``.

The test suite has been expanded substantially around noisy SLOS, g2 behavior, encoding spaces, state-vector inputs, deprecation removals, MerlinProcessor local execution, photonic generators, QCNNs, and QORC reservoir workflows.